# Adding Alzheimer’s molecular subtypes to PrimeKG

This notebook adds **five subtype disease nodes**, connects them to existing gene/protein nodes using published protein measurements, and uses a small GNN to rank drugs.

**Published biology → new subtype nodes → existing PrimeKG proteins → drug candidates.**

Source: [Tijms et al., Nature Aging (2024), Supplementary Table 4](https://www.nature.com/articles/s43587-023-00550-7). These are research molecular subtypes. Different biology may produce different rankings, but that is something we test, not assume.

Keep this notebook beside `ad_primekg.py` and the included `data` folder. The full script and methodological details are in `README.md`. No AD drug predictions have been generated in advance.


## 1. Install in this notebook’s environment
Run once. Restart the kernel if Jupyter asks you to after installation.

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 2. Choose the graph and run name
Set `KG_PATH` to your **original PrimeKG kg.csv**. With downloads enabled, a missing file is downloaded from Harvard Dataverse (approximately 1 GB). Allow several GB of free disk space. The study workbook and HGNC mapping snapshot are already included.

For another experiment, change `RUN_NAME`. Completed exports are not overwritten. To reopen existing results, skip the merge and training cells.


In [2]:
from pathlib import Path
import pandas as pd
import json
import ad_primekg

KG_PATH = Path("../kg_data/kg.csv")# Change this to your existing original PrimeKG file.
DOWNLOAD_IF_MISSING = True
RUN_NAME = "ad_run_01"
GRAPH_DIR = Path("data") / RUN_NAME
RESULTS_DIR = Path("results") / RUN_NAME

print("Working folder:", Path.cwd())
print("Input graph:", KG_PATH.resolve())
print("Merged graph:", GRAPH_DIR.resolve())
print("Results:", RESULTS_DIR.resolve())


Working folder: /Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes
Input graph: /Users/ed/Desktop/Hackathon/kg_data/kg.csv
Merged graph: /Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/data/ad_run_01
Results: /Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/results/ad_run_01


## 3. Create the merged graph
The five groups are neuronal hyperplasticity, innate immune activation, RNA dysregulation, choroid plexus dysfunction, and blood–brain barrier dysfunction.

Defaults retain proteins with q ≤ 0.05 and an absolute difference of at least 0.5 control SD, with sufficient observations. Proteins must map unambiguously to existing PrimeKG gene IDs. Higher and lower CSF protein abundance are different relations. No subtype–drug edges are created.


In [4]:
import importlib
import ad_primekg
importlib.reload(ad_primekg)

<module 'ad_primekg' from '/Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/ad_primekg.py'>

In [5]:
args = ["prepare", "--kg", str(KG_PATH),
        "--supplement", "data/ad_supplement.xlsx",
        "--hgnc", "data/hgnc_complete_set.txt",
        "--out", str(GRAPH_DIR)]
if DOWNLOAD_IF_MISSING:
    args.append("--download")
ad_primekg.main(args)


/Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/ad_primekg.py:122: UserWarning: 126 nodes share a typed external ID. Preserving all node indices; ambiguous gene IDs are excluded from new subtype mappings.
  


                 measured_proteins  significant_proteins  added_gene_edges
subtype_id                                                                
AD:Tijms2024:S1               3863                   318               318
AD:Tijms2024:S2               3863                   657               652
AD:Tijms2024:S3               3863                  1020              1015
AD:Tijms2024:S4               3863                   556               554
AD:Tijms2024:S5               3863                  1198              1186
Saved merged graph to /Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/data/ad_run_01


**Alternative for your earlier graph export:** replace the preceding cell with the following, editing both paths. Do not run both merge versions into the same output folder.

```python
ad_primekg.main([
    "prepare",
    "--nodes", "kg_data/primekg_ecograph/nodes.csv",
    "--edges", "kg_data/primekg_ecograph/edges.csv",
    "--supplement", "data/ad_supplement.xlsx",
    "--hgnc", "data/hgnc_complete_set.txt",
    "--out", str(GRAPH_DIR),
])
```


## 4. Check the new nodes, edges and subtype differences

In [6]:
nodes = pd.read_csv(GRAPH_DIR / "nodes.csv")
evidence = pd.read_csv(GRAPH_DIR / "subtype_gene_evidence.csv")
new_nodes = nodes[nodes["node_id"].astype(str).str.startswith("AD:Tijms2024:")]
assert len(new_nodes) == 5
assert new_nodes["node_type"].eq("disease").all()
assert evidence["x_index"].isin(new_nodes["node_index"]).all()
assert not evidence.duplicated(["x_index", "y_index"]).any()

display(new_nodes[["node_index", "node_id", "node_name", "node_type"]])
display(pd.read_csv(GRAPH_DIR / "merge_summary.csv"))
display(pd.crosstab(evidence["subtype_name"], evidence["relation"]))
display(pd.read_csv(GRAPH_DIR / "subtype_input_cosine.csv", index_col=0).round(3))
# Similarity near 1 means the model receives nearly the same weighted protein inputs.
audit = pd.read_csv(GRAPH_DIR / "gene_mapping_audit.csv")
display(audit.groupby(["subtype_name", "mapping_status"]).size().unstack(fill_value=0))


,node_index,node_id,node_name,node_type
129375,129375,AD:Tijms2024:S1,Alzheimer molecular subtype 1: Neuronal hyperp...,disease
129376,129376,AD:Tijms2024:S2,Alzheimer molecular subtype 2: Innate immune a...,disease
129377,129377,AD:Tijms2024:S3,Alzheimer molecular subtype 3: RNA dysregulation,disease
129378,129378,AD:Tijms2024:S4,Alzheimer molecular subtype 4: Choroid plexus ...,disease
129379,129379,AD:Tijms2024:S5,Alzheimer molecular subtype 5: Blood–brain bar...,disease


,subtype_id,measured_proteins,significant_proteins,added_gene_edges
0,AD:Tijms2024:S1,3863,318,318
1,AD:Tijms2024:S2,3863,657,652
2,AD:Tijms2024:S3,3863,1020,1015
3,AD:Tijms2024:S4,3863,556,554
4,AD:Tijms2024:S5,3863,1198,1186


relation,ad_subtype_higher_csf_protein,ad_subtype_lower_csf_protein
subtype_name,,
Blood–brain barrier dysfunction,429,757
Choroid plexus dysfunction,241,313
Innate immune activation,533,119
Neuronal hyperplasticity,274,44
RNA dysregulation,439,576


,AD:Tijms2024:S1,AD:Tijms2024:S2,AD:Tijms2024:S3,AD:Tijms2024:S4,AD:Tijms2024:S5
subtype_id,,,,,
AD:Tijms2024:S1,1.000,0.505,0.397,0.191,0.284
AD:Tijms2024:S2,0.505,1.000,0.449,0.301,0.328
AD:Tijms2024:S3,0.397,0.449,1.000,0.199,0.302
AD:Tijms2024:S4,0.191,0.301,0.199,1.000,0.438
AD:Tijms2024:S5,0.284,0.328,0.302,0.438,1.000


mapping_status,ambiguous_primekg_gene_id,conflicting_protein_directions,mapped,not_in_primekg,unresolved_identifier
subtype_name,,,,,
Blood–brain barrier dysfunction,4,2,3844,2,11
Choroid plexus dysfunction,4,2,3844,2,11
Innate immune activation,4,8,3838,2,11
Neuronal hyperplasticity,4,2,3844,2,11
RNA dysregulation,4,6,3840,2,11


## 5. Train and evaluate the GNN
The model learns from existing PrimeKG indications. Entire disease groups are held out for validation and testing; identical gene neighborhoods stay together. Indication edges are labels and are **never included in message passing**. No subtype response labels are available.

The GNN pools gene information into diseases and drugs, then learns to score their pairs. This small model uses the relevant disease–gene and drug–gene part of the merged graph. It uses absolute protein-effect weights; the higher/lower direction remains in the saved evidence but is not a treatment-reversal signal.

Default: three initialization seeds, up to 100 epochs, validation-based early stopping. For a quick initial run, use one seed and 30 epochs. CPU training can take a while on the full graph.


In [7]:
ad_primekg.main([
    "train", "--graph", str(GRAPH_DIR), "--out", str(RESULTS_DIR),
    "--epochs", "100", "--seeds", "42", "43", "44",
])


/Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/ad_primekg.py:122: UserWarning: 126 nodes share a typed external ID. Preserving all node indices; ambiguous gene IDs are excluded from new subtype mappings.
  warnings.warn(f'{int(duplicates.sum())} nodes share a typed external ID. '


Seed 42: validation AP 0.2686; test AP 0.2219; popularity 0.1173
Seed 43: validation AP 0.2439; test AP 0.1795; popularity 0.1173
Seed 44: validation AP 0.2539; test AP 0.2264; popularity 0.1173
Rankings and checks saved to /Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/results/ad_run_01


## 6. Did the graph model beat a simple baseline?
These results test recovery of known links for held-out general diseases, **not whether a drug treats an Alzheimer’s subtype**. Missing indications are background/unlabeled, not confirmed ineffective treatments.


In [8]:
metrics = pd.read_csv(RESULTS_DIR / "heldout_disease_metrics.csv")
display(metrics.groupby("seed")[[
    "average_precision", "popularity_average_precision",
    "recall_at_20", "popularity_recall_at_20"
]].mean())
if metrics["average_precision"].mean() <= metrics["popularity_average_precision"].mean():
    print("GNN did not beat drug popularity. Do not present its rankings as validated discoveries.")


,average_precision,popularity_average_precision,recall_at_20,popularity_recall_at_20
seed,,,,
42,0.221876,0.11728,0.352447,0.174669
43,0.179505,0.11728,0.313202,0.174669
44,0.226441,0.11728,0.330337,0.174669


## 7. Did the subtype information actually matter?
A shared-input control gives all subtypes the same representation. A gene-shuffle control changes which genes attach to each subtype. Low overlap demonstrates sensitivity to inputs, not that the new rankings are medically correct. Compare stability across seeds too.


In [9]:
controls = pd.read_csv(RESULTS_DIR / "subtype_controls.csv")
display(controls)
display(pd.read_csv(RESULTS_DIR / "subtype_ranking_overlap.csv"))
if controls["retained_with_shared_input"].eq(controls["top_k"]).all():
    print("All shortlists survived shared input: top-list subtype specificity was NOT demonstrated.")


,seed,subtype_node_index,top_k,retained_with_shared_input,retained_after_gene_shuffle,score_sd_across_drugs
0,42,129375,20,8,1,1.697589
1,42,129376,20,12,4,1.729708
2,42,129377,20,11,4,1.993264
3,42,129378,20,7,6,2.134303
4,42,129379,20,10,2,2.112182
5,43,129375,20,4,0,1.534191
6,43,129376,20,4,0,1.441492
7,43,129377,20,8,6,1.938619
8,43,129378,20,6,2,1.906623
9,43,129379,20,10,3,1.635350


,subtype_a,subtype_b,shared_top_k,jaccard,score_spearman
0,129375,129376,5,0.142857,0.842098
1,129375,129377,0,0.000000,0.757295
2,129375,129378,4,0.111111,0.693700
3,129375,129379,2,0.052632,0.742270
4,129376,129377,3,0.081081,0.761199
5,129376,129378,4,0.111111,0.754519
6,129376,129379,5,0.142857,0.759813
7,129377,129378,5,0.142857,0.896871
8,129377,129379,9,0.290323,0.910514
9,129378,129379,12,0.428571,0.923317


## 8. Show every top drug
**Higher score ranks first.** These are uncalibrated link scores, not AUCs or treatment probabilities. Flags identify known Alzheimer indications and contraindications. A known drug that was already in training is not an independent validation result. Review target coverage and seed stability before choosing candidates.


In [10]:
top = pd.read_csv(RESULTS_DIR / "top_drugs.csv")
columns = ["subtype", "rank", "drug", "drug_id", "ranking_score", "score_sd",
           "seeds_in_top_k", "training_target_coverage", "known_alzheimer_indication",
           "known_alzheimer_indication_in_training", "known_alzheimer_contraindication"]
with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    display(top[columns])
print("All candidates:", (RESULTS_DIR / "all_drug_rankings.csv").resolve())


,subtype,rank,drug,drug_id,ranking_score,score_sd,seeds_in_top_k,training_target_coverage,known_alzheimer_indication,known_alzheimer_indication_in_training,known_alzheimer_contraindication
0,Alzheimer molecular subtype 1: Neuronal hyperplasticity,1,Apomorphine,DB00714,4.323592,1.623915,1,1.0,False,False,False
1,Alzheimer molecular subtype 1: Neuronal hyperplasticity,2,Ziprasidone,DB00246,4.257856,2.325728,2,1.0,False,False,True
2,Alzheimer molecular subtype 1: Neuronal hyperplasticity,3,Diphenhydramine,DB01075,4.166386,1.352234,1,1.0,False,False,False
3,Alzheimer molecular subtype 1: Neuronal hyperplasticity,4,Bromocriptine,DB01200,4.146797,2.681746,1,1.0,False,False,False
4,Alzheimer molecular subtype 1: Neuronal hyperplasticity,5,Quetiapine,DB01224,4.139945,2.175779,1,1.0,False,False,True
5,Alzheimer molecular subtype 1: Neuronal hyperplasticity,6,Amoxapine,DB00543,4.106401,1.923686,1,1.0,False,False,False
6,Alzheimer molecular subtype 1: Neuronal hyperplasticity,7,Pizotifen,DB06153,4.070123,1.340642,0,1.0,False,False,False
7,Alzheimer molecular subtype 1: Neuronal hyperplasticity,8,Desipramine,DB01151,4.069791,1.176510,0,1.0,False,False,False
8,Alzheimer molecular subtype 1: Neuronal hyperplasticity,9,Aripiprazole lauroxil,DB14185,4.048924,2.065682,1,1.0,False,False,False
9,Alzheimer molecular subtype 1: Neuronal hyperplasticity,10,Paliperidone,DB01267,4.045340,2.347483,2,1.0,False,False,True


All candidates: /Users/ed/Desktop/Hackathon/PrimeKG_AD_subtypes/results/ad_run_01/all_drug_rankings.csv


## 9. Inspect the factual protein connections
These paths show what connects a subtype and a drug in the graph. They do not prove the protein is causal, the drug reaches the brain, or that inhibiting it helps. Some neural candidates will not have a direct shared-protein path.


In [11]:
paths = pd.read_csv(RESULTS_DIR / "supporting_paths.csv")
display(paths.head(50))
# Example, after choosing a drug from the ranking:
# display(paths[paths["drug"].str.contains("your drug", case=False, na=False)])


,subtype,drug,drug_id,gene,gene_node_index,subtype_relation,mean_z,q_global_bh,drug_relation,source
0,Alzheimer molecular subtype 1: Neuronal hyperp...,Apomorphine,DB00714,ALB,4315,ad_subtype_lower_csf_protein,-0.501,5.818454e-05,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
1,Alzheimer molecular subtype 1: Neuronal hyperp...,Ziprasidone,DB00246,ALB,4315,ad_subtype_lower_csf_protein,-0.501,5.818454e-05,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
2,Alzheimer molecular subtype 1: Neuronal hyperp...,Aripiprazole lauroxil,DB14185,ALB,4315,ad_subtype_lower_csf_protein,-0.501,5.818454e-05,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
3,Alzheimer molecular subtype 1: Neuronal hyperp...,Risperidone,DB00734,ALB,4315,ad_subtype_lower_csf_protein,-0.501,5.818454e-05,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
4,Alzheimer molecular subtype 1: Neuronal hyperp...,Aripiprazole,DB01238,ALB,4315,ad_subtype_lower_csf_protein,-0.501,5.818454e-05,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
5,Alzheimer molecular subtype 2: Innate immune a...,Aripiprazole,DB01238,ALB,4315,ad_subtype_lower_csf_protein,-0.614,5.651805e-06,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
6,Alzheimer molecular subtype 2: Innate immune a...,Ziprasidone,DB00246,ALB,4315,ad_subtype_lower_csf_protein,-0.614,5.651805e-06,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
7,Alzheimer molecular subtype 2: Innate immune a...,Clozapine,DB00363,GSTP1,1497,ad_subtype_higher_csf_protein,0.587,8.758234e-04,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
8,Alzheimer molecular subtype 2: Innate immune a...,Clozapine,DB00363,CALY,22002,ad_subtype_higher_csf_protein,0.720,1.293603e-07,drug_protein,https://doi.org/10.1038/s43587-023-00550-7
9,Alzheimer molecular subtype 2: Innate immune a...,Apomorphine,DB00714,ALB,4315,ad_subtype_lower_csf_protein,-0.614,5.651805e-06,drug_protein,https://doi.org/10.1038/s43587-023-00550-7


## What you can say in the presentation
“We added five published Alzheimer’s molecular subtype nodes to PrimeKG using measured protein differences. We trained a graph model on existing drug–disease knowledge and tested whether the new subtype information changes candidate rankings.”

Only add that the model beats popularity, produces subtype-sensitive lists, or gives stable candidates if the checks above support those statements. Different-looking lists alone are not a discovery. Drug mechanisms, brain exposure, contradictory evidence and independent experiments remain to be assessed.
